# Changes and Updates in TE-Agents2

This notebook documents the significant updates and improvements made by Noah Zeidenberg to `TEWorldCodeV2` as a research student co-supervised by Dr. Stefan Kremer and T. Ryan Gregory at the University of Guelph. The changes encompass migration to modern Python standards, major performance optimizations, and functional enhancements to the biological simulation.

## 1. Python 3 Migration and Modern Syntax

The codebase has been fully ported from Python 2.7 to Python 3.x.

*   **Print Function:** Replaced `print "..."` statements with `print(...)` functions.
*   **Exception Handling:** Updated exception syntax from `except Exception, e:` to `except Exception as e:`.
*   **Type Hinting:** Extensive use of Python type hints (e.g., `List`, `Dict`, `Optional`, `int`) throughout the code to improve readability and IDE support.
*   **Modern Imports:** Integration of standard libraries for concurrency and typing (`concurrent.futures`, `typing`).

### Example: Print Statements and Exception Handling

In [ ]:
# Original (Python 2.7) - TEWorldCodeV2/TESim.py
def output( keyword, message ):
  if not parameters.output.has_key(keyword) or parameters.output[keyword]:
    print "[%s]:%s" % (keyword,message);

try:
  self.elements.remove(element);
except ValueError, e:
  print ">>>325>>>", element;
  raise;

In [ ]:
# New (Python 3) - TE-Agents2/ABM_files/TESim_ABM2.py
def output(keyword: str, message: str):
    # ... logging logic ...
    logger.log(log_level, f"[{keyword}]: {message}")

# Exception handling in modern syntax (example from context, though logic changed)
try:
    # ...
except Exception as e:
    output("GENE INIT", f"Exception during gene creation: {e}")

## 2. Performance Optimizations

Significant effort has been put into optimizing the simulation for speed and memory efficiency, enabling larger-scale simulations and longer timescales.

### Advanced Data Structures: IntervalTree

*   **IntervalTree:** Chromosome element lookups (finding what exists at a specific genomic location) now use an `IntervalTree` structure, offering $O(\log n)$ complexity compared to the previous $O(n)$ linear search.

In [ ]:
# Original - Linear Search (TEWorldCodeV2/TESim.py)
def __getitem__( self, index ):
  """
  Returns the element at the given index, or Junk if no element exists
  at the index.
  """
  # could make this more efficient via binary search
  for element in self.elements:
    if element.start <= index < element.end: # start of element in region
      return element;
  return Junk;

In [ ]:
# New - IntervalTree Lookup (TE-Agents2/ABM_files/TESim_ABM2.py)
def __getitem__(self, index: int):
    """
    Fast element lookup using IntervalTree; O(log n) instead of O(n).
    Returns the element at the given index, or JUNK_TYPE if no element exists.
    """
    # Use bit array for very fast checking in large genomes
    if self.occupancy_bits and not self.occupancy_bits.get_bit(index):
        return JUNK_TYPE
    
    # IntervalTree lookup
    overlapping = self.interval_tree.at(index)
    if overlapping:
        return next(iter(overlapping)).data  # Return first overlapping element
    return JUNK_TYPE

### Vectorization and JIT Compilation

*   **Numpy & Numba:** Core calculations (fitness effects, survival probabilities) now use `numpy` for vectorized operations instead of Python loops.
*   **JIT Compilation:** Critical "hot paths" are optimized using `numba`'s `@njit` decorator.

In [ ]:
# Original - Loop-based Selection (TEWorldCodeV2/TESim.py)
def selection_and_drift( self ):
  total_fitness = sum( [ i.fitness for i in self.individual ]);
  if total_fitness > 0.0:
    new_population = [ i for i in self.individual 
                      if random.random() < parameters.Host_survival_rate( i.fitness/total_fitness ) ];
  else:
    new_population = [];

  self.individual = new_population;

In [ ]:
# New - Vectorized Selection (TE-Agents2/ABM_files/TESim_ABM2.py)
def selection_and_drift(self):
    # ...
    # Vectorized fitness calculation
    fitnesses = np.array([ind.fitness for ind in self.individual])
    total_fitness = np.sum(fitnesses)
    
    if total_fitness > 0.0:
        # Vectorized survival probability calculation
        survival_probs = batch_survival_probability(
            fitnesses, total_fitness, parameters.Carrying_capacity
        )
        
        # Vectorized random selection
        random_vals = vrng.uniform(size=len(self.individual))
        survivors_mask = random_vals < survival_probs
        
        # Filter survivors
        self.individual = [ind for i, ind in enumerate(self.individual) 
                         if survivors_mask[i]]

## 3. Functional Biological Enhancements

The biological realism of the simulation has been expanded with new element types and behaviors.

### Gene Subtypes and Structure
*   **Gene Subtypes:** Unlike the single `ProkGene1` in V2, the new code supports multiple eukaryotic gene subtypes: `ORF` (protein-coding), `promoter`, `enhancer`, `intron`, `silencer`, and `insulator`. A TE insertion in an intron could have a lesser impact on host fitness than in an ORF, for example.
*   **Eukaryotic Structure:** Genes now model internal structure (introns/exons).

In [ ]:
# Original - Simple Prokaryotic Gene (TEWorldCodeV2/TESim.py)
class ProkGene1(Element):
  """
  Subclass of Element.  Simple model of a prokaryotic gene which has no 
  introns and a fixed gene length of 1600 BPs.
  """
  length = parameters.Gene_length;
  
  def __init__( self, start ):
    Element.__init__( self, self.length, start );

In [ ]:
# New - Eukaryotic Gene with Subtypes (TE-Agents2/ABM_files/TESim_ABM2.py)
class EukGene(Element):
    """Memory-efficient eukaryotic gene with introns, exons, and regulatory regions."""
    
    def __init__(self, start: int, length: int = None, subtype: str = None):
        # ... initialization logic ...
        super().__init__(length, start, GENE_TYPE, subtype)
        
        # Initialize eukaryotic-specific properties
        self._initialize_eukaryotic_structure()
    
    def _initialize_eukaryotic_structure(self):
        """Initialize eukaryotic gene structure based on subtype."""
        if self.subtype == 'ORF':
            # Protein-coding gene with introns and exons
            self.exon_count = max(1, int(self.length / 1000))
            self.intron_count = max(0, self.exon_count - 1)
            self.has_splice_sites = self.intron_count > 0
            # ...
        elif self.subtype == 'promoter':
            # ... promoter logic ...

Note that diploidy, meiosis, recombination, mating and fertalization mechanics have not been tested, and are excluded from the current stable release of `TE-Agents2`.

### Transposable Element (TE) Diversity

*   **TE Families:** The simulation now supports specific TE families (e.g. `SINE`, `LINE`, `LTR`, `DNA_TRANSPOSON`, ...) with distinct characteristics. These characteristics can be modified to more accurately represent specific classes, families, clades, *etc*.
*   **Autonomous vs. Non-Autonomous:** Explicit modeling of autonomous TEs and non-autonomous TEs (like SINEs) that require "helpers". This optional mechanism is helpful in simulating predator-prey oscillations, red-queen dynamics, competition and driver vs. passenger dynamics.

In [ ]:
# Original - Single TE Type (TEWorldCodeV2/TESim.py)
class SelectiveInsertTE(Element):
  # ...
  def jump( self ):
    # ... basic jump logic ...
    if parameters.TE_excision_rate==0.0:  # assume retro-transposon
      progeny = parameters.TE_progeny.generate();
    elif random.random()<parameters.TE_excision_rate:
      self.chromosome.excise( self );
      progeny = parameters.TE_progeny.generate();

In [ ]:
# New - Diverse TEs with Autonomy (TE-Agents2/ABM_files/TESim_ABM2.py)
class SelectiveInsertTE(Element):
    # ...
    def can_transpose(self) -> bool:
        """
        Check if this TE can transpose based on autonomous status and parasitism rules.
        Non-autonomous TEs require specific autonomous TEs to be present and active.
        """
        if self.dead:
            return False
        
        if self.autonomous:
            return True
        
        # Non-autonomous TEs need specific autonomous TEs to be present
        if self.chromosome and self.chromosome.host:
            # ... parasitism logic ...
            parasitism_targets = parameters.get_te_parasitism_targets(self.te_type)
            # ... check for targets ...

## 4. Configuration and Logging

*   **YAML Configuration:** Parameters are no longer hardcoded in a `.py` file but are loaded from a flexible `example_config.yaml` file, allowing for easy experiment setup without code modification. High-low parameter testing can easily be initiated with a SLURM array job.
*   **Enhanced Logging:** A robust logging system allows for configurable output levels (INFO, DEBUG, WARNING) and directs output to both console and files. If you encounter issues running the script please include debug logs in your issue submission.
*   **Optimized Tracefiles:** Data collection writes to `trace.csv` in buffered batches to reduce I/O overhead, and dynamically adds columns for new TE types and gene subtypes.

In [ ]:
# Original - Hardcoded Parameters (TEWorldCodeV2/parameters.py)
Gene_length = 1000;
TE_length = 1000;
TE_death_rate = 0.5;
TE_excision_rate = 0.1;
Initial_genes = 500;

In [ ]:
# New - YAML Configuration Loading (TE-Agents2/ABM_files/parameters_ABM2.py)
def load_config(config_file="example_config.yaml"):
    """Load configuration from YAML file with fallback to defaults"""
    with open(config_file, 'r') as fh:
        config = yaml.safe_load(fh)
    return config

# ...
simulation_config = config.get('simulation', {})
TE_death_rate = simulation_config.get('te_death_rate', 0.5)
Initial_genes = simulation_config.get('initial_genes', 500)